# Dialogues protocolisés : inquiry et persuasion (Walton–Krabbe)

Carnet fondé sur le carnet de référence du sas EPITA `dialogue_protocols`
(`docs/coursia_contrib`), qui fait autorité pour son contenu, et sur sa **fixture
partagée** — neuf transitions et sept historiques de verdicts attendus, rangés dans
`data/dialogue_protocols_examples.json` (exemples synthétiques domaine public).
Le module Python `dialogue_protocols.py` définit les machines de référence. Ce carnet les implémente en C# et confronte leurs verdicts à la fixture commune.

**Idée directrice.** Un protocole de dialogue est une *machine à états sur des actes
de langage* : chaque coup est un acte parmi neuf (affirmer, questionner, contester,
argumenter, concéder, retirer, soutenir, réfuter, comprendre) ; chaque enchaînement
est permis ou interdit par une table de transitions ; et la fin du dialogue n'est pas
une impression mais une condition testable. Deux protocoles suffisent à couvrir deux
natures opposées de l'échange : l'*inquiry*, coopératif, qui cherche un accord sur
les faits, et la *persuasion*, adversarial, où l'un des deux doit céder.


**Statut et provenance.** Les trois classes `DialogueProtocol` /
`InquiryProtocol` / `PersuasionProtocol` que ce carnet enseigne ont ete
**retirees du tronc amont** par #2137 (2026-09-17). Le tronc EPITA declare
mot pour mot : *"dead twins — the living workflow path for formal dialogue
is the JVM `logic/dialogue_handler.py`"* ; leurs seuls utilisateurs
restants y sont *"the CoursIA teaching notebooks"*. Le module
`dialogue_protocols.py` documente ce statut (lignes 19-32) et assume la
distillation comme **contenu pedagogique executable sans JVM**. Le pendant
de production est `argumentation_analysis/agents/core/logic/dialogue_handler.py`
du tronc EPITA ([DialogueHandler.execute_dialogue](https://github.com/jsboigeEpita/2025-Epita-Intelligence-Symbolique/blob/ecfd9b9c31d299ad9e08ee0e7a726d436f4ebd8e/argumentation_analysis/agents/core/logic/dialogue_handler.py#L33)),
un pont `jpype` vers Tweety (ici porté par IKVM vers .NET) qui execute un dialogue proponent/opponent
reel : fusion des vues du cadre de Dung, extension *fondée* via
`SimpleGroundedReasoner`, verdict `accepted`/`rejected` sur le sujet
debattu. Ce carnet n'est donc pas *le* moteur de la maison : c'est la
seule implementation vivante de cette abstraction, sous forme de C# BCL, et il le dit.

**Objectifs :** valider les transitions et arrêts, comparer un verdict Tweety fondé. **Prérequis :** C#, .NET Interactive et IKVM. **Durée :** 90 minutes.

**Série Argumentation** | | Précédent : [03d-ADF-Semantiques-Python](./Argumentation-03d-ADF-Semantiques-Python.ipynb) | Suivant : [04b-Knowledge-Base-CSharp](./Argumentation-04b-Knowledge-Base-CSharp.ipynb) | [↑ Série Argumentation](README.md)

## Pourquoi protocoliser un dialogue ?

Walton et Krabbe (*Commitment in Dialogue*, 1995) typent les dialogues par leur but :
chercher de l'information, enquêter, persuader, négocier, délibérer, ou simplement
s'affronter. Un dialogue laissé libre ne sait dire ni qu'il a mal tourné, ni qu'il
est fini : une conversation qui tourne en rond et une conversation qui conclut se
ressemblent, vues de l'extérieur, tant qu'aucune règle ne les distingue.

Protocoliser, c'est rendre ces deux jugements **mécaniques** :

1. **Un coup illégal a une position.** La table de transitions dit, pour chaque
   acte, quels actes peuvent légitimement suivre. Une contestation — « tu n'avais
   pas le droit de concéder ici » — se tranche en consultant la table, pas en
   argumentant sur les intentions.
2. **La fin est une condition, pas un sentiment.** Compréhension mutuelle,
   capitulation, impasse, longueur maximale : le protocole énumère ses façons de
   mourir, et coupe lui-même les dialogues qui tournent en rond.
3. **Le dialogue devient reproductible.** Deux traces identiques reçoivent le même
   verdict : le dialogue devient l'objet d'une expérience, pas seulement d'une
   conversation.

Le registre Walton–Krabbe déclare six types de dialogue ; deux seulement ont une
machine. Ce n'est pas un défaut à corriger mais un état de la recherche : une
typologie nomme plus qu'elle ne construit, et l'écart entre les deux est en soi
instructif — c'est la première chose que mesure ce carnet.

Note de portee. Le pendant *de production* d'un dialogue protocolise
est `DialogueHandler` (Tweety/JVM) dans le tronc amont — voir l'encart de
la cellule d'introduction pour le statut de cette distillation et le chemin
vivant.

In [1]:
// Machine locale : tables explicites et ordre des conditions de terminaison.
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Text.Json;

enum DialogueType
{
    INFORMATION_SEEKING,
    INQUIRY,
    PERSUASION,
    NEGOTIATION,
    DELIBERATION,
    ERISTIC
}

enum SpeechAct
{
    CLAIM,
    QUESTION,
    CHALLENGE,
    ARGUE,
    CONCEDE,
    RETRACT,
    SUPPORT,
    REFUTE,
    UNDERSTAND
}

record Move(string Speaker, SpeechAct Act, string Content = "", string Target = null);
record TraceResult(bool Valid, int? FirstInvalid, bool Terminated, string Reason);
class Protocol
{
    public string Name;
    public Dictionary<SpeechAct, SpeechAct[]> Transitions;
    public Protocol(string name, Dictionary<SpeechAct, SpeechAct[]> transitions)
    {
        Name = name;
        Transitions = transitions;
    }

    public SpeechAct[] Allowed(SpeechAct a) => Transitions.TryGetValue(a, out var acts) ? acts : Array.Empty<SpeechAct>();
    public bool IsValid(SpeechAct a, SpeechAct b) => Allowed(a).Contains(b);
    public string Reason(IReadOnlyList<Move> h)
    {
        int n = h.Count;
        if (Name == "inquiry")
        {
            if (n >= 3 && h.Skip(n - 3).All(x => x.Act == SpeechAct.UNDERSTAND || x.Act == SpeechAct.CONCEDE))
                return "_term_mutual_understanding";
            if (n > 25)
                return "_term_length";
            if (n >= 4 && h[n - 1].Act == SpeechAct.UNDERSTAND && h[n - 2].Act == SpeechAct.UNDERSTAND)
                return "_term_converged";
            if (n >= 6 && h[n - 6].Act == h[n - 4].Act && h[n - 4].Act == h[n - 2].Act && h[n - 5].Act == h[n - 3].Act && h[n - 3].Act == h[n - 1].Act)
                return "_term_loop";
        }
        else
        {
            if (n > 0 && h[n - 1].Act == SpeechAct.CONCEDE)
                return "_term_concede";
            if (n > 30)
                return "_term_length";
            if (n > 2 && h[n - 1].Act == SpeechAct.RETRACT && h[n - 2].Act == SpeechAct.RETRACT)
                return "_term_double_retract";
        }

        return null;
    }

    public TraceResult Validate(IReadOnlyList<Move> h)
    {
        int? invalid = null;
        for (int i = 1; i < h.Count; i++)
            if (!IsValid(h[i - 1].Act, h[i].Act))
            {
                invalid = i;
                break;
            }

        var reason = Reason(h);
        return new(invalid == null, invalid, reason != null, reason);
    }
}

var inq = new Protocol("inquiry", new() { [SpeechAct.QUESTION] = new[] { SpeechAct.CLAIM, SpeechAct.ARGUE, SpeechAct.QUESTION, SpeechAct.SUPPORT }, [SpeechAct.CLAIM] = new[] { SpeechAct.SUPPORT, SpeechAct.CHALLENGE, SpeechAct.QUESTION, SpeechAct.UNDERSTAND }, [SpeechAct.SUPPORT] = new[] { SpeechAct.QUESTION, SpeechAct.UNDERSTAND, SpeechAct.CHALLENGE }, [SpeechAct.CHALLENGE] = new[] { SpeechAct.ARGUE, SpeechAct.SUPPORT, SpeechAct.QUESTION }, [SpeechAct.ARGUE] = new[] { SpeechAct.CHALLENGE, SpeechAct.UNDERSTAND, SpeechAct.QUESTION, SpeechAct.SUPPORT }, [SpeechAct.UNDERSTAND] = new[] { SpeechAct.QUESTION, SpeechAct.UNDERSTAND, SpeechAct.CLAIM }, [SpeechAct.REFUTE] = new[] { SpeechAct.ARGUE, SpeechAct.QUESTION }, [SpeechAct.CONCEDE] = new[] { SpeechAct.QUESTION, SpeechAct.UNDERSTAND } });
var per = new Protocol("persuasion", new() { [SpeechAct.CLAIM] = new[] { SpeechAct.CHALLENGE, SpeechAct.CONCEDE, SpeechAct.QUESTION, SpeechAct.SUPPORT }, [SpeechAct.CHALLENGE] = new[] { SpeechAct.ARGUE, SpeechAct.RETRACT, SpeechAct.SUPPORT }, [SpeechAct.ARGUE] = new[] { SpeechAct.CHALLENGE, SpeechAct.CONCEDE, SpeechAct.REFUTE, SpeechAct.SUPPORT }, [SpeechAct.QUESTION] = new[] { SpeechAct.CLAIM, SpeechAct.ARGUE, SpeechAct.SUPPORT }, [SpeechAct.REFUTE] = new[] { SpeechAct.ARGUE, SpeechAct.CONCEDE, SpeechAct.CHALLENGE }, [SpeechAct.SUPPORT] = new[] { SpeechAct.UNDERSTAND, SpeechAct.CHALLENGE, SpeechAct.QUESTION }, [SpeechAct.CONCEDE] = new[] { SpeechAct.CLAIM, SpeechAct.QUESTION }, [SpeechAct.RETRACT] = new[] { SpeechAct.CLAIM, SpeechAct.QUESTION }, [SpeechAct.UNDERSTAND] = new[] { SpeechAct.CLAIM, SpeechAct.QUESTION } });
var protos = new Dictionary<string, Protocol>
{
    ["inquiry"] = inq,
    ["persuasion"] = per
};
Console.WriteLine($"Types Walton-Krabbe : {Enum.GetValues<DialogueType>().Length} ; actes : {Enum.GetValues<SpeechAct>().Length} ; protocoles : {protos.Count} ; transitions inquiry : {inq.Transitions.Values.Sum(x => x.Length)} ; persuasion : {per.Transitions.Values.Sum(x => x.Length)}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Types Walton-Krabbe : 6 ; actes : 9 ; protocoles : 2 ; transitions inquiry : 25 ; persuasion : 26


### Lecture du compte

Le registre déclare six types — `INFORMATION_SEEKING`, `INQUIRY`, `PERSUASION`,
`NEGOTIATION`, `DELIBERATION`, `ERISTIC` — mais seuls `inquiry` et `persuasion`
ont une machine : quatre types sont des noms sans protocole. Les deux machines sont
d'une taille quasi identique (25 transitions contre 26) alors que leurs natures
s'opposent : la différence entre coopératif et adversarial n'est pas une question
de volume mais de *composition* — c'est ce que lira le tableau croisé de fin de
carnet, ligne par ligne.

## Les neuf actes de parole

La pragmatique linguistique classe les actes de langage par ce qu'ils *font* plutôt
que par ce qu'ils disent. Le registre des actes en formalise neuf, que l'on peut
grouper en quatre familles :

| Famille | Actes | Effet sur l'engagement |
|---|---|---|
| Assertifs | `CLAIM`, `SUPPORT` | le locuteur s'engage sur un contenu |
| Directifs | `QUESTION`, `CHALLENGE` | le locuteur exige quelque chose de l'interlocuteur — une réponse, une justification |
| Argumentatifs | `ARGUE`, `REFUTE` | le locuteur attaque ou défend un contenu par des raisons |
| Conclusifs | `CONCEDE`, `RETRACT`, `UNDERSTAND` | le locuteur clôt un point : il cède, retire sa parole, ou déclare avoir compris |

La donnée que manipule la machine n'est ni la phrase ni sa syntaxe : c'est l'acte.
Deux phrases différentes qui contestent — « pourquoi cela ? » et « sur quoi repose
votre chiffre ? » — sont le même coup de `CHALLENGE`. C'est ce nivellement qui rend
l'état d'un dialogue dénombrable : la machine ne compte pas des mots, elle compte
des coups.

In [2]:
// Comparer les réponses et les actes qui mènent à CONCEDE.
foreach (var p in protos.Values)
{
    Console.WriteLine($"Après QUESTION ({p.Name}) : {string.Join(", ", p.Allowed(SpeechAct.QUESTION))}");
    Console.WriteLine($"Vers CONCEDE : {string.Join(", ", Enum.GetValues<SpeechAct>().Where(a => p.IsValid(a, SpeechAct.CONCEDE)))}");
}

Après QUESTION (inquiry) : CLAIM, ARGUE, QUESTION, SUPPORT


Vers CONCEDE : 


Après QUESTION (persuasion) : CLAIM, ARGUE, SUPPORT


Vers CONCEDE : CLAIM, ARGUE, REFUTE


### Lecture : deux tables, deux mondes

Après `QUESTION`, l'inquiry autorise… `QUESTION` à nouveau : relancer la
clarification est légitime dans une enquête. La persuasion l'exclut — après une
question, on répond (`CLAIM`, `ARGUE`, `SUPPORT`), on ne questionne pas.

Plus frappant : `CONCEDE` n'est atteignable par **aucun** acte en inquiry —
l'acte de capitulation existe dans l'énumération des neuf, mais aucune transition
n'y mène : c'est un acte mort dans ce protocole. En persuasion, trois actes y
mènent (`CLAIM`, `ARGUE`, `REFUTE`). La capitulation est une porte qui n'existe
que dans le monde adversarial.

## Le verdict attendu : la fixture partagée

Les tables ci-dessus disent ce que le moteur **fait** ; la fixture partagée dit ce
qu'il **doit** faire. Neuf transitions et sept historiques y sont consignés avec
leur verdict attendu et sa raison — c'est la source de vérité commune du carnet de
référence et de la garde aller-retour `tests/test_dialogue_protocols.py`. Passer les
tables au crible de la fixture, c'est séparer la spécification de l'implémentation :
si un jour le moteur diverge, c'est ici que le désaccord aura un lieu.

In [3]:
// Vérification des transitions à partir de la fixture suivie par le dépôt.
var examples = JsonDocument.Parse(File.ReadAllText("data/dialogue_protocols_examples.json")).RootElement;
var transitions = examples.GetProperty("transitions").EnumerateArray().ToArray();
int bonnes = 0;
foreach (var e in transitions)
{
    var protocol = protos[e.GetProperty("dialogue").GetString()];
    var a = Enum.Parse<SpeechAct>(e.GetProperty("from").GetString());
    var b = Enum.Parse<SpeechAct>(e.GetProperty("to").GetString());
    bool got = protocol.IsValid(a, b);
    if (got == e.GetProperty("allowed").GetBoolean())
        bonnes++;
    Console.WriteLine($"{protocol.Name}: {a} -> {b} : {got} ; {e.GetProperty("why").GetString()}");
}

Console.WriteLine($"Transitions conformes : {bonnes}/{transitions.Length}");

inquiry: QUESTION -> CLAIM : True ; En inquiry, une question peut recevoir une affirmation directe.


inquiry: CLAIM -> CONCEDE : False ; Asymétrie clé : en inquiry on ne concède pas une affirmation — on la soutient, la conteste ou la questionne.


inquiry: CHALLENGE -> ARGUE : True ; Une contestation appelle un argument.


inquiry: UNDERSTAND -> UNDERSTAND : True ; La compréhension mutuelle peut s'enchaîner.


persuasion: CLAIM -> CONCEDE : True ; En persuasion au contraire, concéder une affirmation est la sortie normale — l'objectif du dialogue.


persuasion: CHALLENGE -> RETRACT : True ; Contesté, le locuteur peut retirer sa position.


persuasion: ARGUE -> REFUTE : True ; Un argument peut être réfuté de front.


persuasion: QUESTION -> CHALLENGE : False ; Une question n'appelle pas une contestation directe — d'abord une réponse (claim/argue/support).


persuasion: CONCEDE -> ARGUE : False ; Après une concession, on reformule une position ou on questionne, on ne relance pas un argument.


Transitions conformes : 9/9


### Lecture : la spécification a un fichier

Chaque ligne porte son verdict **et sa raison** — l'asymétrie la plus instructive
est `CLAIM -> CONCEDE` : interdit en inquiry (« on ne concède pas une affirmation
qu'on cherche à établir ensemble »), autorisé en persuasion (« concéder est la
fin normale du débat »). Le même couple d'actes, deux mondes : ce n'est pas le
moteur qui le décide, c'est la fixture qui l'exige.

In [4]:
// Les états terminaux sont indépendants de la validité de leur chemin.
var terminations = examples.GetProperty("terminations").EnumerateArray().ToArray();
int bons = 0;
foreach (var e in terminations)
{
    var protocol = protos[e.GetProperty("dialogue").GetString()];
    var hist = e.GetProperty("acts").EnumerateArray().Select((a, i) => new Move($"loc{i % 2}", Enum.Parse<SpeechAct>(a.GetString()))).ToList();
    bool got = protocol.Reason(hist) != null;
    if (got == e.GetProperty("terminal").GetBoolean())
        bons++;
    Console.WriteLine($"{protocol.Name}: {string.Join(" -> ", hist.Select(x => x.Act))} : {got} ; {e.GetProperty("why").GetString()}");
}

Console.WriteLine($"Historiques conformes : {bons}/{terminations.Length}");

inquiry: UNDERSTAND -> UNDERSTAND -> UNDERSTAND : True ; Trois compréhensions/consécutions consécutives en fin d'historique : dialogue clos.


inquiry: CLAIM -> QUESTION -> UNDERSTAND -> UNDERSTAND : True ; Deux compréhensions consécutives en position 4 : condition de convergence atteinte.


inquiry: CLAIM -> SUPPORT -> CLAIM -> SUPPORT -> CLAIM -> SUPPORT : True ; Boucle de motif (claim, support) répétée trois fois : le dialogue tourne en rond, on coupe.


inquiry: QUESTION -> CLAIM -> SUPPORT -> CHALLENGE -> ARGUE : False ; Historique sain et mixte : aucune condition de terminaison ne s'applique.


persuasion: CLAIM -> CHALLENGE -> ARGUE -> CONCEDE : True ; Une concession en dernière position termine la persuasion — l'objectif est atteint.


persuasion: CLAIM -> CHALLENGE -> RETRACT -> RETRACT : True ; Deux retraits consécutifs : plus rien à persuader.


persuasion: CLAIM -> CHALLENGE -> ARGUE : False ; Persuasion en cours, pas de concession ni retrait final.


Historiques conformes : 7/7


### Lecture : une entrée à double fond

L'historique `CHALLENGE -> RETRACT -> RETRACT` est consigné **TERMINAL** par la
fixture — et pourtant la table de transitions interdit d'y arriver : seul
`CHALLENGE` mène à `RETRACT`, et `RETRACT` ne mène pas à `RETRACT`. La condition
de terminaison `_term_double_retract` vit donc sur un état que aucun dialogue
légal ne peut produire. Ce n'est pas une erreur de la fixture : elle teste
l'état, pas la légalité du chemin — et le croisement des deux tables met au jour
une propriété du moteur que la section finale mesure en détail.

## La trace, unité d'observation

L'objet que manipule le validateur n'est pas le dialogue en train de se dérouler mais
sa *trace* : la liste des coups joués, dans l'ordre. Chaque coup est un
`DialogueMove` — un locuteur, un acte, un contenu libre et une cible optionnelle.
La machine à états ne lit que l'acte : le contenu sert à l'humain qui relit la
trace, pas au protocole.

`Validate(moves)` rend quatre choses : la validité transition par transition
(`valid`), l'index du premier coup illégal s'il y en a un (`first_invalid`), la
terminaison (`terminated`) et la condition de terminaison satisfaite (`reason`).
C'est l'instrument unique de ce carnet : chaque expérience qui suit se lit dans ces
quatre champs.

In [5]:
// Trace d'enquête aboutissant à la compréhension mutuelle.
List<Move> Trace(params SpeechAct[] acts) => acts.Select((a, i) => new Move(i % 2 == 0 ? "A" : "B", a)).ToList();
var traceInq = new List<Move>
{
    new("A", SpeechAct.QUESTION, "Le seuil de 80 % est-il atteint ?"),
    new("B", SpeechAct.CLAIM, "Oui, la mesure du jour donne 82 %"),
    new("A", SpeechAct.SUPPORT, "J’ajoute : hier on était déjà à 81 %"),
    new("B", SpeechAct.UNDERSTAND, "Nos deux mesures se corroborent"),
    new("A", SpeechAct.UNDERSTAND, "Compris, nous sommes d’accord sur le fait")
};
Console.WriteLine($"{string.Join(" -> ", traceInq.Select(x => x.Act))} : {inq.Validate(traceInq)}");

QUESTION -> CLAIM -> SUPPORT -> UNDERSTAND -> UNDERSTAND : TraceResult { Valid = True, FirstInvalid = , Terminated = True, Reason = _term_converged }


### Lecture : conclure sans gagner

Les cinq coups sont légitimes (`valid: True`, `FirstInvalid` vide) et le
dialogue est terminé par `_term_converged` : deux `UNDERSTAND` consécutifs. Personne
n'a cédé, personne n'a gagné : les deux parties déclarent avoir compris la même
chose. C'est la signature de l'inquiry — l'accord sur les faits, pas la victoire
d'une thèse. Noter que la condition ne regarde que la fin de la trace : les deux
derniers coups suffisent, le reste de l'historique est indifférent à cette
terminaison-là.

In [6]:
// Un motif de période 2 est légal mais la boucle termine l'échange.
var traceLoop = Trace(Enumerable.Range(0, 3).SelectMany(_ => new[] { SpeechAct.QUESTION, SpeechAct.CLAIM }).ToArray());
Console.WriteLine(inq.Validate(traceLoop));

TraceResult { Valid = True, FirstInvalid = , Terminated = True, Reason = _term_loop }


### Lecture : légal et mort à la fois

`valid: True` **et** `terminated: True` : chaque transition est permise, et
pourtant le protocole déclare le dialogue terminé (`_term_loop`). La même paire
d'actes `(QUESTION, CLAIM)` répétée trois fois est irréprochable coup par coup et
vide dans son ensemble. C'est la limite de la validité transitionnelle — elle ne
voit que les enchaînements locaux — et la raison pour laquelle la terminaison est
une couche *séparée* de la table : détecter qu'un dialogue ne progresse plus
demande de regarder la forme globale de la trace, pas seulement ses arêtes.

In [7]:
// Le détecteur d'origine ne détecte pas la période 3.
var traceP3 = Trace(Enumerable.Range(0, 3).SelectMany(_ => new[] { SpeechAct.QUESTION, SpeechAct.CLAIM, SpeechAct.SUPPORT }).ToArray());
Console.WriteLine(inq.Validate(traceP3));

TraceResult { Valid = True, FirstInvalid = , Terminated = False, Reason =  }


### Lecture : la limite mesurée du détecteur

Le motif `(QUESTION, CLAIM, SUPPORT)` répété trois fois passe : `terminated:
False`, aucune raison de terminaison. Neuf coups qui tournent en rond traversent
le filtre, alors que six coups en période 2 étaient coupés juste avant. La cause
est dans la définition du détecteur : il compare les *paires* adjacentes des six
derniers coups — `recent[0:2] == recent[2:4] == recent[4:6]` — et une période 3 ne
fait jamais coïncider ces paires. C'est la divergence 3 du port, mesurée ici sur
une trace exécutée ; l'exercice 1 fait écrire le détecteur de période 3 qui
manque.

## Exercices

Trois exercices, un par acquisition. Les stubs s'exécutent sans erreur (règle C.1) et la méthode de chaque exercice est déjà démontrée sur un exemple dans le carnet :

- **Exercice 1** (détecteur de période 3) → la cellule « motif 3 » l'a montré en sortie : `valid : True` mais `terminated : False` avec `Reason` vide — le motif de période 3 est légal et ne meurt d'aucune des conditions de terminaison ; l'indice `motif[0:3] == motif[3:6] == motif[6:9]` transpose le geste de la démo.
- **Exercice 2** (persuasion de 31 coups) → les cellules de terminaison ont montré les deux morts à éviter : `reason : _term_concede` (sur `CLAIM -> CHALLENGE -> ARGUE -> CONCEDE`) et `_term_double_retract` (sur `CHALLENGE -> RETRACT -> RETRACT`) ; la boucle de quatre actes se vérifie arête par arête avec `IsValid`, comme la fixture l'a fait sur ses 9 transitions.
- **Exercice 3** (qui meurt, et comment) → la fixture a énuméré les causes possibles (`_term_converged`, `_term_loop`, `_term_concede`, `_term_length`) ; la simulation seedée mesure leur distribution, inquiry contre persuasion — on la lance pour savoir, pas pour confirmer.


## Exercice 1 — le détecteur de période 3

Le détecteur de boucle du protocole ne voit que les répétitions de *paires* d'actes
(période 2). Écrire `DetectPeriod3(history)` qui rend `True` si et seulement
si les neuf derniers coups forment trois répétitions d'un même triplet d'actes, et
`False` sinon.

Témoins attendus :

- la trace `traceP3` de la démonstration ci-dessus doit rendre `True` ;
- la trace `traceLoop` (période 2 : `(QUESTION, CLAIM)` répété) doit rendre
  `False` ;
- une trace de moins de neuf coups doit rendre `False`.

Indications : extraire les neuf derniers actes dans une liste `motif`, puis
comparer le triplet de tête aux deux suivants — `motif[0:3] == motif[3:6] ==
motif[6:9]`.

In [8]:
// Exercice 1 : reconnaître les neuf derniers actes comme trois triplets identiques.
bool? DetectPeriod3(IReadOnlyList<Move> history)
{ /* TODO étudiant : comparer les trois triplets ; moins de neuf coups donne false. */
    return null;
}

Console.WriteLine(DetectPeriod3(traceP3)?.ToString() ?? "Exercice à compléter");

Exercice à compléter


## Persuasion : le même formalisme, une autre nature

Le passage de l'inquiry à la persuasion ne change ni le formalisme ni l'instrument :
mêmes neuf actes, même `Validate`. Ce qui change est la *table* — et à travers
elle, la nature de l'échange. L'inquiry est coopératif : deux parties construisent
un accord sur les faits, et personne ne « gagne ». La persuasion est adversariale :
deux thèses s'affrontent, et le dialogue se conclut par une capitulation —
`CONCEDE` — ou par épuisement.

Où se lit l'adversarialité ? Dans les transitions elles-mêmes. `REFUTE` est vivant
en persuasion (on peut attaquer une thèse, et l'attaque appelle une riposte) ; il
est quasi muet en inquiry (on demande, on clarifie, on ne détruit pas). `RETRACT`
n'existe qu'en persuasion : dans une enquête on ne retire pas sa parole, on la
précise. La table de transitions est donc une théorie du dialogue *encodée en
machine* — les expériences qui suivent la lisent coup par coup.

In [9]:
// Persuasion : la concession termine la trace.
var tracePer = new List<Move>
{
    new("A", SpeechAct.CLAIM, "La dessalinisation est la meilleure option pour la ville"),
    new("B", SpeechAct.CHALLENGE, "Pourquoi celle-là plutôt que le recyclage ?"),
    new("A", SpeechAct.ARGUE, "Coût par m3 le plus bas à l’échelle de nos besoins"),
    new("B", SpeechAct.CONCEDE, "Soit : à cette échelle, j’accepte l’argument")
};
Console.WriteLine(per.Validate(tracePer));

TraceResult { Valid = True, FirstInvalid = , Terminated = True, Reason = _term_concede }


### Lecture : la victoire codée

`reason: _term_concede` — le dernier coup est une capitulation, et cela suffit.
Comparée à la trace d'inquiry (conclure par `UNDERSTAND`), la différence tient en
un acte : le même formalisme, un `CONCEDE` à la place d'un `UNDERSTAND`, et le
sens de la fin change — victoire d'une thèse contre accord sur un fait. Aucune
comptabilité de score n'est nécessaire : la condition de terminaison *est* le
verdict.

In [10]:
// Double retrait : état terminal mais transition interdite.
Console.WriteLine($"Vers RETRACT : {string.Join(",", Enum.GetValues<SpeechAct>().Where(a => per.IsValid(a, SpeechAct.RETRACT)))} ; après RETRACT : {string.Join(",", per.Allowed(SpeechAct.RETRACT))}");
var impasse = Trace(SpeechAct.CHALLENGE, SpeechAct.RETRACT, SpeechAct.RETRACT);
Console.WriteLine(per.Validate(impasse));

Vers RETRACT : CHALLENGE ; après RETRACT : CLAIM,QUESTION


TraceResult { Valid = False, FirstInvalid = 2, Terminated = True, Reason = _term_double_retract }


### Lecture : une condition de terminaison morte

La trace `CHALLENGE -> RETRACT -> RETRACT` est **invalide à l'index 2**
(`RETRACT` n'admet que `CLAIM` et `QUESTION`), et pourtant
Le résultat a `Terminated = true` et `Reason` donne `_term_double_retract`. La condition est
satisfaite par une trace qu'aucun dialogue légal ne peut produire. En effet, seul
`CHALLENGE` mène à `RETRACT`, et `RETRACT` ne mène pas à `RETRACT` : deux
retraits consécutifs sont un état interdit par la table. La condition vit dans le
code et reste morte dans le dialogue — c'est la divergence 5 du port, découverte
en écrivant cette démonstration. Leçon de spécification : une machine à états se
vérifie aussi du côté de ses conditions de terminaison — chacune doit correspondre
à un état *atteignable* par les transitions permises.

In [11]:
// La même transition change de statut selon le protocole.
var bad = Trace(SpeechAct.REFUTE, SpeechAct.CONCEDE);
Console.WriteLine($"inquiry : {inq.Validate(bad)} ; persuasion : {per.Validate(bad)}");

inquiry : TraceResult { Valid = False, FirstInvalid = 1, Terminated = False, Reason =  } ; persuasion : TraceResult { Valid = True, FirstInvalid = , Terminated = True, Reason = _term_concede }


### Lecture : illégal relativement à quoi ?

`REFUTE -> CONCEDE` est rejeté à l'index 1 en inquiry : une réfutation appelle
des raisons (`ARGUE`) ou une précision (`QUESTION`), jamais une capitulation. La
même trace, exécutée sur la table de persuasion, est **légale** : là, `REFUTE`
admet explicitement `CONCEDE` — détruire un argument peut faire céder
l'adversaire. Un coup n'est donc pas illégal dans l'absolu : il est illégal
*relativement à un protocole*. C'est tout ce que « protocoliser » achète : la
contestation « tu n'avais pas le droit de conclure ici » reçoit une réponse
mécanique, tranchée par la table et localisée à l'index près.

In [12]:
// Tableau complet de toutes les réponses autorisées.
foreach (var a in Enum.GetValues<SpeechAct>())
    Console.WriteLine($"{a, -12} | {string.Join(",", inq.Allowed(a)), -40} | {string.Join(",", per.Allowed(a))}");

CLAIM        | SUPPORT,CHALLENGE,QUESTION,UNDERSTAND    | CHALLENGE,CONCEDE,QUESTION,SUPPORT


QUESTION     | CLAIM,ARGUE,QUESTION,SUPPORT             | CLAIM,ARGUE,SUPPORT


CHALLENGE    | ARGUE,SUPPORT,QUESTION                   | ARGUE,RETRACT,SUPPORT


ARGUE        | CHALLENGE,UNDERSTAND,QUESTION,SUPPORT    | CHALLENGE,CONCEDE,REFUTE,SUPPORT


CONCEDE      | QUESTION,UNDERSTAND                      | CLAIM,QUESTION


RETRACT      |                                          | CLAIM,QUESTION


SUPPORT      | QUESTION,UNDERSTAND,CHALLENGE            | UNDERSTAND,CHALLENGE,QUESTION


REFUTE       | ARGUE,QUESTION                           | ARGUE,CONCEDE,CHALLENGE


UNDERSTAND   | QUESTION,UNDERSTAND,CLAIM                | CLAIM,QUESTION


## Exercice 2 — une persuasion qui ne meurt jamais (presque)

Construire une trace **légale** de persuasion de 31 coups qui atteint la condition
`_term_length` (plus de 30 coups) sans jamais satisfaire ni `_term_concede` ni
`_term_double_retract` : autrement dit, un dialogue où personne ne cède et personne
ne retire deux fois de suite, jusqu'à épuisement du temps imparti.

La fonction `TracePersuasionLongue()` doit renvoyer la liste des `DialogueMove`.
Le témoin est `Validate` : `valid` doit valoir `True` et `reason` doit valoir
`_term_length`.

Indication : il existe un cycle de quatre actes légal en persuasion, par exemple
`CHALLENGE -> SUPPORT -> QUESTION -> CLAIM -> CHALLENGE -> ...` — vérifier chaque
arête avec `IsValid` avant d'écrire la boucle.

In [13]:
// Exercice 2 : 31 coups de persuasion légaux, sans concession ni double retrait.
List<Move> TracePersuasionLongue()
{ /* TODO étudiant : choisir un cycle légal et alterner A/B. */
    return null;
}

Console.WriteLine(TracePersuasionLongue() == null ? "Exercice à compléter" : per.Validate(TracePersuasionLongue()).ToString());

Exercice à compléter


### Lecture croisée : la nature se lit dans la table

Quatre différences structurelles sautent du tableau :

1. **`RETRACT` est muet en inquiry, vivant en persuasion** — dans une enquête on
   précise sa parole, on ne la retire pas ; dans un affrontement, se rétracter est
   un coup tactique.
2. **`UNDERSTAND` boucle sur lui-même en inquiry** (`UNDERSTAND` est dans ses
   propres successeurs — c'est le mécanisme exact de `_term_converged`), **pas en
   persuasion** où il ne mène qu'à `CLAIM` ou `QUESTION` : comprendre est une fin
   en enquête, un simple tour de passe en dispute.
3. **`SUPPORT` ouvre sur `UNDERSTAND` en persuasion** mais sur `QUESTION` en
   inquiry : soutenir y appelle une réaction immédiate, ici une clarification.
4. **`CONCEDE` est la cible de trois actes en persuasion, d'aucun en inquiry** —
   la porte de sortie existe à chaque étage du monde adversarial, et n'existe
   nulle part dans le monde coopératif.

La nature d'un dialogue n'est donc pas un attribut déclaratif : c'est la
composition de sa table — et les colonnes ci-dessus en sont la radiographie.

## La sémantique fondée : un second contrat

La validité locale des transitions ne décide pas qui a raison. L'adaptateur du tronc fusionne les vues des deux agents, calcule l'extension fondée avec **Tweety SimpleGroundedReasoner**, puis construit séparément une trace d'assertion, attaques et défenses. Ici la DLL Tweety suivie par le dépôt est exécutée via IKVM 8.15 ; aucune simulation du raisonneur ni appel LLM. Deux cadres confrontent la dessalinisation au recyclage puis ajoutent l'empreinte carbone.

In [14]:
// Tweety réel : DLL Dung compilée par IKVM, sans repli sur un moteur local.
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "../Tweety/org.tweetyproject.tweety-dung.dll"
using org.tweetyproject.arg.dung.syntax;
using org.tweetyproject.arg.dung.reasoner;
using org.tweetyproject.arg.dung.semantics;
using System.Reflection;
using System.Runtime.InteropServices;

string ver = "8.15.0", rid = (OperatingSystem.IsWindows() ? "win" : OperatingSystem.IsMacOS() ? "osx" : "linux") + "-" + RuntimeInformation.ProcessArchitecture.ToString().ToLowerInvariant();
string nr = Environment.GetEnvironmentVariable("NUGET_PACKAGES") ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string baseDir = Path.Combine(nr, "ikvm.image", ver, "ikvm", "any", "any"), archDir = Path.Combine(nr, "ikvm.image.runtime." + rid, ver, "ikvm", "any", rid), home = Path.Combine(Path.GetTempPath(), "ikvm-dialogue-home-" + ver + "-" + rid);
void CopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var dest = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(dest));
        File.Copy(f, dest, true);
    }
}

if (!Directory.Exists(baseDir) || !Directory.Exists(archDir))
    throw new InvalidOperationException("Image IKVM requise absente");
Directory.CreateDirectory(home);
CopyMerge(baseDir, home);
CopyMerge(archDir, home);
AppContext.SetData("IKVM.Home", home);
var assembly = AssemblyName.GetAssemblyName("../Tweety/org.tweetyproject.tweety-dung.dll");
Console.WriteLine($"Tweety : {assembly.Name} {assembly.Version} ; IKVM {ver}");
// Le raisonneur calcule l'extension ; la trace suit séparément l'adaptateur source.
void ExecuteDialogue(string[] propArgs, (string, string)[] propAtt, string[] oppArgs, (string, string)[] oppAtt, string topic, int maxRounds = 10)
{
    var names = propArgs.Concat(oppArgs).Distinct().ToArray();
    var allAtt = propAtt.Concat(oppAtt).Distinct().ToArray();
    var theory = new DungTheory();
    var args = names.ToDictionary(n => n, n => new Argument(n));
    foreach (var a in args.Values)
        theory.add(a);
    foreach (var(a, b)in allAtt)
        if (args.ContainsKey(a) && args.ContainsKey(b))
            theory.add(new Attack(args[a], args[b]));
    var ext = new SimpleGroundedReasoner().getModel(theory);
    var accepted = ext.toArray().Cast<Argument>().Select(x => x.getName()).OrderBy(x => x).ToArray();
    Console.WriteLine($"Cadre : {string.Join(",", names.OrderBy(x => x))} ; attaques : {string.Join(",", allAtt.Select(x => $"{x.Item1}->{x.Item2}"))}");
    Console.WriteLine($"Extension fondée : {string.Join(",", accepted)} ; verdict : {(accepted.Contains(topic) ? "accepted" : "rejected")}");
    var trace = new List<string>
    {
        $"1 proponent assert {topic}"};
    int round = 2;
    foreach (var(a, b)in oppAtt)
        if (b == topic || propAtt.Any(x => x.Item1 == b))
        {
            trace.Add($"{round} opponent attack {a}->{b}");
            round++;
            if (round > maxRounds)
                break;
        }

    foreach (var(a, b)in propAtt)
        foreach (var(c, d)in oppAtt)
            if (b == c)
            {
                trace.Add($"{round} proponent defend {a}->{b}");
                round++;
                if (round > maxRounds)
                    break;
            }

    Console.WriteLine($"Trace ({trace.Count} rounds) : {string.Join(" ; ", trace)}");
}

ExecuteDialogue(new[] { "dessalinisation", "cout_m3" }, new[] { ("cout_m3", "recyclage") }, new[] { "recyclage" }, new[] { ("recyclage", "dessalinisation") }, "dessalinisation");
ExecuteDialogue(new[] { "dessalinisation", "cout_m3" }, new[] { ("cout_m3", "recyclage") }, new[] { "recyclage", "empreinte_carbone" }, new[] { ("recyclage", "dessalinisation"), ("empreinte_carbone", "dessalinisation") }, "dessalinisation");

Installed Packages IKVM, 8.15.0 IKVM.Image, 8.15.0

Tweety : org.tweetyproject.tweety-dung 1.30.0.0 ; IKVM 8.15.0


Cadre : cout_m3,dessalinisation,recyclage ; attaques : cout_m3->recyclage,recyclage->dessalinisation


Extension fondée : cout_m3,dessalinisation ; verdict : accepted


Trace (3 rounds) : 1 proponent assert dessalinisation ; 2 opponent attack recyclage->dessalinisation ; 3 proponent defend cout_m3->recyclage


Cadre : cout_m3,dessalinisation,empreinte_carbone,recyclage ; attaques : cout_m3->recyclage,recyclage->dessalinisation,empreinte_carbone->dessalinisation


Extension fondée : cout_m3,empreinte_carbone ; verdict : rejected


Trace (4 rounds) : 1 proponent assert dessalinisation ; 2 opponent attack recyclage->dessalinisation ; 3 opponent attack empreinte_carbone->dessalinisation ; 4 proponent defend cout_m3->recyclage


### Lecture : légalité locale contre sort global

Le coût sans attaquant défend la dessalinisation contre le recyclage. L'argument carbone non attaqué fait sortir la thèse de l'extension fondée. La trace est assemblée par l'adaptateur après le calcul Tweety : elle n'est pas une simulation des états intermédiaires du raisonneur.

### Les neuf actes, côté production : un canal déclaré sans producteur

Le vocabulaire Walton-Krabbe de ce carnet a bien eu un répondant de production — mais
**dans le tronc EPITA, pas dans ce dépôt**. Dans
`argumentation_analysis/agents/core/informal/dung_arbitration_stage.py` du dépôt
[`2025-Epita-Intelligence-Symbolique`](https://github.com/jsboigeEpita/2025-Epita-Intelligence-Symbolique),
la constante `WALTON_KRABBE_ATTACKING_ACTS` séparait les actes attaquants (`REFUTE`,
`CHALLENGE` — ceux qui deviennent des attaques Dung) des actes coopératifs (`CONCEDE`,
`SUPPORT`, `UNDERSTAND`), et `walton_krabbe_conflict_policy` transformait les relations
déclarées entre candidats en attaques.

Deux réserves mesurées, avant de lire ce rapprochement comme un « usage de production ».

**Ce canal n'est pas porté ici.** `dung_arbitration_stage.py` ne figure pas dans le
`NOTICE-EPITA` de `argumentation_lib/` : aucune copie verbatim n'en a été vendue dans
CoursIA. Le dialogue de grounding exécuté plus haut, lui, **est** porté
(`_dialogue_handler.py`) — c'est l'écart entre citer un organe et en exécuter un.

**Et il n'existe plus en amont.** Le 2026-10-05, le commit `166a5e7c` l'a retiré avec
un motif qui vaut leçon : « retire the Walton-Krabbe relations channel, whose producer
never existed ». Le canal lisait `context["walton_krabbe_relations"]` alors que
**personne n'écrivait jamais cette clé** ; `DebateAgent` produit des scores de qualité,
pas des actes `REFUTE`/`CHALLENGE`.

C'est l'anti-modèle exact de ce que ce carnet enseigne : une politique qui transforme
des relations que nul ne fournit reste cohérente tant qu'on ne demande pas **qui les
produit**. Les neuf actes gardent ici leur valeur de **spécification exécutable** —
table de transitions, condition de terminaison, dialogue de grounding réellement
exécuté — et non celle d'une politique branchée sur un flux inexistant.


## Jonction avec les assets voisins : le schéma d'un argument du dialogue

Le protocole échange des *actes* ; l'argument lui-même a une structure. Deux assets
voisins de la série la portent : le classifieur de schémas de Walton
(`argumentation_schemes.py`, carnet `01b`) reconnaît, sur son texte, le schéma d'un
argument — appel à l'expert, exemple, cause… — et la base de connaissances
(`knowledge_base.py`, carnet `04b`) structure prémisses, conclusion et schéma dans
un `FormalArgument`. Le carnet de référence joint les trois niveaux : un argument
du dialogue dont le champ `scheme` est peuplé par le classifieur.

Le pendant formel de l'echange lui-meme — pas seulement de l'argument
unitaire — est `DialogueHandler` (Tweety/JVM) cote production : la
trace d'un dialogue execute se materialise par un verdict `accepted`/
`rejected` sur le sujet debattu, et c'est ce qui rend le `scheme` ci-dessus
non seulement descriptible mais *decidable*.

In [15]:
// Table lexicale complète et ordre canonique du module voisin ; aucune étiquette fabriquée.
var catalogue = new (string Key, double Strength, string[][] Keywords)[]
{
    ("expert_opinion", 0.8, new[] { new[] { "expert", "domaine" }, new[] { "selon", "spécialiste" }, new[] { "autorité", "compétent" }, new[] { "source", "expert" } }),
    ("empirical_evidence", 0.9, new[] { new[] { "données", "représentatif" }, new[] { "étude", "mesure" }, new[] { "statistique", "échantillon" }, new[] { "résultat", "expérience" } }),
    ("consensus", 0.85, new[] { new[] { "consensus", "experts" }, new[] { "majorité", "accord" }, new[] { "unanime", "scientifiques" } }),
    ("analogy", 0.6, new[] { new[] { "analogue", "similaire" }, new[] { "comme", "semblable" }, new[] { "à l'instar", "comparable" } }),
    ("cause_effect", 0.7, new[] { new[] { "cause", "effet" }, new[] { "entraîne", "conséquence" }, new[] { "provoque", "résultat" }, new[] { "donc", "parce que" } }),
    ("economic_argument", 0.75, new[] { new[] { "coût", "bénéfice" }, new[] { "économique", "rentable" }, new[] { "investissement", "retour" } }),
    ("precautionary_principle", 0.7, new[] { new[] { "risque", "prévention" }, new[] { "principe de précaution" }, new[] { "danger", "éviter" } }),
    ("moral_argument", 0.8, new[] { new[] { "droit", "violation" }, new[] { "morale", "devoir" }, new[] { "éthique", "injuste" }, new[] { "droits", "atteinte" } }),
    ("historical_precedent", 0.65, new[] { new[] { "précédent", "historique" }, new[] { "déjà", "passé" }, new[] { "autrefois", "abouti" } }),
    ("modus_ponens", 1.0, new[] { new[] { "donc", "implique" }, new[] { "par conséquent", "si" } })
};
(string Key, double Strength)? Classify(string text)
{
    var lower = text.ToLowerInvariant();
    foreach (var s in catalogue)
        if (s.Keywords.Any(group => group.All(lower.Contains)))
            return (s.Key, s.Strength);
    return null;
}

record DialogueProposition(string Content, double Confidence = 1.0);
class DialogueArgument
{
    public List<DialogueProposition> Premises;
    public DialogueProposition Conclusion;
    public string Scheme;
    public string Id;
    public DialogueArgument(IEnumerable<DialogueProposition> premises, DialogueProposition conclusion, string scheme, string id = null)
    {
        Premises = premises.ToList();
        Conclusion = conclusion;
        Scheme = scheme;
        Id = string.IsNullOrEmpty(id) ? Guid.NewGuid().ToString() : id;
    }

    public override string ToString() => $"[{string.Join(", ", Premises.Select(x => x.Content))}] -> {Conclusion.Content}";
}

var texte = "Selon un chercheur spécialiste du domaine, la méthode est fiable.";
var scheme = Classify(texte);
var argument = new DialogueArgument(new[] { new DialogueProposition("Le chercheur est spécialiste du domaine", .9) }, new DialogueProposition("La méthode est fiable", .8), scheme?.Key);
Console.WriteLine($"Argument : {argument} ; schéma : {argument.Scheme}, force : {scheme?.Strength:F2}, identifiant : {argument.Id[..8]}...");

Argument : [Le chercheur est spécialiste du domaine] -> La méthode est fiable ; schéma : expert_opinion, force : 0.80, identifiant : bb3a9684...


### Lecture : trois niveaux, une seule chaîne

Le classifieur rend un `ArgumentationScheme` (`expert_opinion`, force a priori),
la base range l'argument (`[premisses] -> conclusion`) et le protocole, lui, dira
quand cet argument peut être *posé* (`CLAIM`), *soutenu* (`SUPPORT`) ou *réfuté*
(`REFUTE`). Le champ `scheme` est la clé de jonction : c'est elle qu'une
démonstration automatique lirait pour relier le tour de parole à la théorie de
l'argumentation qu'il instancie.

## Propriétés mesurées du moteur

Le moteur est volontairement **documenté plutôt que corrigé** : ses angles morts
sont mesurables, et chacun enseigne ce qu'un protocole bien spécifié doit vérifier.

1. **Six types déclarés, deux machines.** Le registre Walton–Krabbe énumère six
   types de dialogue ; seuls `INQUIRY` et `PERSUASION` ont un protocole implémenté.
   Une typologie nomme plus qu'elle ne construit — l'écart est en soi instructif.
2. **La boucle de période 3 n'est pas vue.** Le détecteur de boucle compare les
   paires d'actes adjacentes sur les six derniers coups : un motif de période 2
   répété est coupé, un motif de période 3 traverse le filtre (démonstration plus
   haut, exercice 1).
3. **Limites de longueur asymétriques.** 25 coups pour l'inquiry, 30 pour la
   persuasion, sans justification documentée — portées telles quelles.
4. **La condition `_term_double_retract` est inatteignable.** Elle teste deux
   `RETRACT` consécutifs — un état que la table de transitions interdit : seul
   `CHALLENGE` mène à `RETRACT`, et `RETRACT` ne mène pas à `RETRACT`. La fixture
   partagée le met en évidence sans le nommer : elle consigne l'historique
   `CHALLENGE -> RETRACT -> RETRACT` comme TERMINAL (la condition tire sur
   l'état), alors qu'aucun dialogue légal ne peut l'atteindre (la table interdit
   le chemin). Condition vive sur l'état, morte par la table : du code que seule
   la croisée des deux spécifications fait voir.

Documenter plutôt que corriger est ici une décision d'ingénierie : l'essence d'un
moteur de protocole, ce sont aussi ses angles morts. Ils enseignent ce qu'un
protocole bien spécifié doit vérifier — ici, que chaque condition de terminaison
corresponde à un état réellement atteignable par les transitions permises.

## Exercice 3 — qui meurt, et comment ?

Simuler 200 dialogues par protocole et compter les causes de mort. Deux locuteurs
A et B alternent ; chaque coup est tiré au hasard (générateur seedé) parmi les
réponses permises au dernier acte ; le dialogue démarre sur `QUESTION` (inquiry) ou
`CLAIM` (persuasion) et s'arrête quand `Terminated` vaut `true`. `Reason` fournit alors le libellé de terminaison. La fonction
`SimulerDialogues(protocole, n, seed)` doit renvoyer un dictionnaire
`{raison_de_terminaison: compte}`.

Questions auxquelles la distribution répondra : l'inquiry meurt-elle plutôt par
compréhension mutuelle ou par longueur ? La persuasion cède-t-elle (`_term_concede`)
plus souvent qu'elle ne s'épuise (`_term_length`) ? La réponse n'est pas intuitive —
c'est le propre d'une expérience : on la lance pour savoir, pas pour confirmer.

In [16]:
// Exercice 3 : simuler 200 dialogues avec Random(seed+i) et compter les causes.
Dictionary<string, int> SimulerDialogues(Protocol protocole, int n = 200, int seed = 0)
{ /* TODO étudiant */
    return null;
}

Console.WriteLine(SimulerDialogues(inq) == null ? "Exercice à compléter" : "Distribution calculée");

Exercice à compléter


## Vérifiez votre compréhension

Pourquoi la validité locale ne suffit-elle pas à décider l'acceptation d'une thèse ? Pourquoi le motif de période 3 échappe-t-il au détecteur ?

<details><summary>Éléments de réponse</summary>

La validité teste les transitions ; Tweety calcule l'extension fondée du cadre fusionné. Le détecteur de boucle ne compare que trois paires d'actes.
</details>

## À retenir

Ce carnet a lu deux protocoles de dialogue comme ce qu'ils sont : des machines à
états sur des actes de langage. Trois choses à retenir :

1. **La table de transitions EST la théorie.** Coopératif ou adversarial ne se
   décrivent pas en adjectifs : `REFUTE` vivant ou muet, `RETRACT` présent ou
   absent, `CONCEDE` cible de trois actes ou d'aucun — la nature du dialogue se
   lit dans la table, coup par coup. Mesuré : `Transitions inquiry : 25` contre `26` en persuasion ; `REFUTE -> CONCEDE` est `valid : False` en inquiry (`first_invalid : 1`, REFUTE n'admet que ARGUE et QUESTION) mais `valid = True` en persuasion ; CONCEDE est ciblée par `['CLAIM', 'ARGUE', 'REFUTE']` en persuasion et par **aucun** acte en inquiry.
2. **La terminaison se spécifie, sinon elle se décrète.** Le protocole coupe les
   dialogues qui tournent en rond et énumère ses morts ; la divergence 5 enseigne
   la réciproque — une condition de terminaison écrite pour un état que la table
   interdit est du code mort, qu'on ne découvre qu'en cherchant à le déclencher — mesuré : `CHALLENGE -> RETRACT -> RETRACT` est `valid : False` (le second RETRACT est illégal) et pourtant `is_terminal_state : True`.
3. **Le formalisme est léger, les conséquences ne le sont pas.** Neuf actes et une
   vingtaine de transitions suffisent à rendre une contestation tranchable, une
   terminaison garantie et une expérience reproductible — la trace `QUESTION -> CLAIM -> SUPPORT -> UNDERSTAND -> UNDERSTAND` conclut sans gain (`terminated : True`, `reason : _term_converged`).

Dans la série, ce carnet complète ses voisins : les notebooks de Toulmin
structurent l'argument unitaire, les Agentic orchestrent le travail collectif
d'agents, la gouvernance multi-agents décide collectivement — ici, c'est l'échange
lui-même qui est procéduralisé, acte par acte. La question « comment parle-t-on ? »
précède la question « que conclut-on ? » : le protocole est ce qui rend la seconde
posée honnêtement.

**Statut et perenite.** Les trois machines que ce carnet enseigne sont
un snapshot du retrait tronc #2137 (2026-09-17) ; le chemin de production
est `logic/dialogue_handler.py` (Tweety/JVM). La distillation est assumee
comme contenu pédagogique de machine à états exécutable sans JVM ; le cas fondé emploie la DLL Tweety via IKVM — sa portee est l'enseignement
de la machine a etats, pas l'execution de dialogue reel. Le present carnet
est l'un des derniers lieux ou ce code mort-vivant est maintenu ; tant qu'il
existe comme organe d'enseignement, les divergences documentees (periode 3,
`_term_double_retract` inatteignable, asymetrie 25/30) restent visibles aux
eleves qui etudient la theorie des protocoles.

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place des protocoles de dialogue dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

Le carnet [03d (ADF)](Argumentation-03d-ADF-Semantiques-Python.ipynb) généralise l'acceptabilité au-delà de l'attaque binaire ; le carnet [04b (base de connaissances)](Argumentation-04b-Knowledge-Base-Python.ipynb) montre la mémoire où les actes de ces protocoles déposent leurs arguments ; le carnet [05 (vérification formelle)](Argumentation-05-Formal-Verification-Python.ipynb) confie les verdicts à un vrai solveur ; la [gouvernance multi-agents](Argument_Analysis_Gouvernance_Multi_Agents.ipynb) replace la prise de parole dans un collectif.

**Référence** — D. N. Walton, E. C. Krabbe (1995). *Commitment in Dialogue: Basic Concepts of Interpersonal Reasoning*, SUNY Press — la typologie des six types de dialogue et l'engagement (commitment) qui fonde les protocoles reconstructeurs.
